# Imports

In [ ]:
import datetime
import ipywidgets as widgets
from IPython.display import display
import re
import time

from collections import OrderedDict
from ipywidgets import interact, interactive
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.signal as signal
import tomli_w
import tomli

# Functions

In [ ]:
def plot_signal(
    data: pd.DataFrame, 
    sample_interval: float = 2
) -> tuple[plt.figure, plt.axis]:
    if data is None: return None
    
    fig, ax = plt.subplots(figsize=(15,8))
    ax.plot(
        np.arange(0, data.shape[0] * sample_interval, sample_interval), 
        data
    )

    ax.grid(visible=True)
    ax.tick_params(axis='both', labelsize=14)
    
    ax.set_xlabel("time (ns)", fontsize=18)
    ax.set_ylabel("amplitude ($V$)", fontsize=18)
    
    return fig, ax


def plot_bases(
    signal: pd.Series, 
    peak_idx: int,
    peak_height: float,
    left_idx: int,
    left_idx_user: int,
    right_idx: int,
    right_idx_user: int,
    sample_interval: float = 2
) -> tuple[plt.figure, plt.axis]:
    
    if signal is None:
        return None
    
    fig, ax = plot_signal(signal, sample_interval)

    # Plot Peak
    ax.plot(
        peak_idx * sample_interval, 
        peak_height, 
        'rx'
    )
    
    # Left Base
    ax.plot(
        left_idx * sample_interval,
        signal[left_idx] - 0.02,
        'r^',
        label="Peak Onset (SciPy)"
    )
    
    
    ax.plot(
        left_idx_user * sample_interval,
        signal[left_idx_user] + 0.02,
        'gv',
        label="Peak Onset (User)"
    )
    
    # Right Base
    ax.plot(
        right_idx * sample_interval,
        signal[right_idx] - 0.02,
        'r^',
        label="Tail Onset (SciPy)"
    )
    
    ax.plot(
        right_idx_user * sample_interval,
        signal[right_idx_user] + 0.02,
        'gv',
        label="Tail Onset (User)"
    )
    
    ax.set_title(f"Plot of Signal {signal.name}")
    ax.legend()

In [ ]:
def generate_psd(
    df: pd.DataFrame, left_bases: dict, right_bases: dict, amplitudes: dict) -> pd.DataFrame:
    def process(series: pd.Series) -> dict:
        q_total = series[left_bases[series.name]:].sum()
        q_tail = series[right_bases[series.name]:].sum()

        res = {
            "q_total": q_total,
            "q_tail": q_tail,
            "quotient": q_tail/q_total
        }

        return res
    
    psd_report = pd.DataFrame({signal_id: process(df[signal_id]) for signal_id in df.columns})
    psd_report.loc["amplitude"] = amplitudes
    return psd_report

# Pre-Processing

In [ ]:
EXP_ROOT = Path("../sample_datasets/20220824_CERC_background/processed_data/cleaned_buffers/")
PARQ_PATH = EXP_ROOT / "20220824-0003_clean.parquet"

In [ ]:
df = pd.read_parquet(PARQ_PATH)
df.columns = df.columns.astype("int16")
df = df.T

## Normalization

In [ ]:
# Normalize
df_norm = (df - df.min()) / (df.max() - df.min())

In [ ]:
plot_signal(df_norm)
plt.show()

## Smoothing

In [ ]:
# signal.savgol_filter()

In [ ]:
df_processed = df_norm

# Pulse Shape Discrimination

## Peak Finding
Since we already found peaks in the `data_cleaning.ipynb` notebook, we can just reuse the settings to determine the same values! In a more complete notebook, we can just reuse these values instead of needing to recompute any information using `scipy.signal.find_peaks()`

In [ ]:
SETTINGS_PATH = EXP_ROOT / "settings.toml"

with open(SETTINGS_PATH, "rb") as f:
    exp_info = tomli.load(f)

multipeak_filter_settings = exp_info["multipeak_filter_settings"]
height = multipeak_filter_settings["height"]
prominence = multipeak_filter_settings["prominence"]

output = df_processed.apply(lambda x: signal.find_peaks(x, height=height, prominence=prominence))
peak_idx, props = output.iloc[0,:], output.iloc[1,:]

### Visualize Peak Finding

In [ ]:
def get_bases(series: pd.Series, peak_idx: int, peak_offset: int=10, tail_offset=5):
    return get_left_bases(series, peak_idx, peak_offset), get_right_bases(series, peak_idx, tail_offset)

def get_left_bases(series: pd.Series, peak_idx: int, peak_offset: int=0):
    return series[peak_idx - peak_offset: peak_idx].idxmin()

def get_right_bases(series: pd.Series, peak_idx: int, tail_offset: int=0):
    # return series[peak_idx: peak_idx + tail_onset].idxmin()
    return peak_idx + tail_offset

In [ ]:
signal_id_dropdown = widgets.Dropdown(options=sample_ids) 
tail_onset_box = widgets.BoundedIntText(value=5, min=1, max=30)
peak_offset_box = widgets.BoundedIntText(value=20, min=1, max=30)

signal_id_dropdown.observe(update_sliders, "value")

def peak_plot_interactable(signal_id, peak_offset, tail_onset):
    output = df_processed.apply(lambda x: get_bases(x, peak_idx[x.name][0], peak_offset, tail_onset))
    left_bases, right_bases = output.iloc[0,:], output.iloc[1,:]

    plot_bases(
        df_processed.get(signal_id), 
        peak_idx[signal_id],
        props[signal_id]["peak_heights"],
        props[signal_id]["left_bases"],  # from scipy.signal.find_peaks()
        left_bases[signal_id],  # from us
        props[signal_id]["right_bases"],  # from scipy.signal.find_peaks()
        right_bases[signal_id],  # from us
    )

    return left_bases, right_bases

peak_plot_interactable = interactive(
    peak_plot_interactable, 
    signal_id = signal_id_dropdown, 
    peak_offset = peak_offset_box, 
    tail_onset = tail_onset_box
)

display(peak_plot_interactable)

## Integration

**ASSUMPTION:** We will take our `left_bases` and `right_bases` result for this integration.

In [ ]:
start_time = time.perf_counter()

left_bases, right_bases = peak_plot_interactable.result
psd_report = generate_psd(df, left_bases, right_bases, df.max())

print(
    f"Generated PSD report in [\x1b[1;32m{(time.perf_counter() - start_time)*1000:.2f} ms\x1b[0m]."
)

In [ ]:
psd_report

## Visualizations

In [ ]:
label_fs = 14

### Tail ($Q_\text{tail}$) vs Total ($Q_\text{total}$)
Here we calculate the ratio of tail/total, $Q$.

In [ ]:
fig1, ax1 = plt.subplots(figsize=(12,9))

ax1.scatter(
    psd_report.loc["q_total"],
    psd_report.loc["q_tail"],
    marker=".",
    s=1
)

ax1.set_xlabel("total integral (a.u.)", fontsize=label_fs)
ax1.set_ylabel("tail integral (a.u.)", fontsize=label_fs)

plt.show()

### Peak Amplitude vs $Q$

In [ ]:
fig2, ax2 = plt.subplots(figsize=(12,9))

ax2.scatter(
    psd_report.loc["amplitude"],
    psd_report.loc["quotient"],
    marker=".",
    s=1
)

ax2.set_xlabel("pulse amplitude ($V$)", fontsize=label_fs)
ax2.set_ylabel("tail / peak (a.u.)", fontsize=label_fs)


plt.show()

### <span style="color:#FF9900">Figure of Merit</span>

In [ ]:
fig3, ax3 = plt.subplots(figsize=(12,9))

n_bins = 100
ax3.hist(
    psd_report.loc["quotient"],
    bins=n_bins,
    linewidth=2,
    histtype='step',
    orientation='horizontal'
)

ax3.text(
    x=0.75,
    y=0.95,
    s=f"n_bins = {n_bins}",
    transform=ax3.transAxes
)

# ax3.set_ylim([0, 1])

ax3.set_xlabel("counts", fontsize=label_fs)
ax3.set_ylabel("tail / total (a.u.)", fontsize=label_fs)

plt.show()

## Report

In [ ]:
# ALL IN ONE INTERACTIVE PLOT
def plot_psd(psd_report: pd.DataFrame, nbins:int = 100):
    fig, axs = plt.subplots(1, 3, figsize=(18,6))
    
    # Q_tail vs Q_tot
    axs[0].scatter(
        psd_report.loc["q_total"],
        psd_report.loc["q_tail"],
        marker='.',
        s=1
    )

    axs[0].set_xlabel("total integral (a.u.)", fontsize=label_fs)
    axs[0].set_ylabel("tail integral (a.u.)", fontsize=label_fs)

    
    # Q_tot vs Amplitude
    axs[1].scatter(
        psd_report.loc['amplitude'],
        psd_report.loc["quotient"],
        marker='.',
        s=1
    )

    axs[1].set_xlabel("pulse amplitude (V)", fontsize=label_fs)
    axs[1].set_ylabel("tail / peak (a.u.)", fontsize=label_fs)

    
    # Q distribution
    axs[2].hist(
        psd_report.loc["quotient"],
        bins=n_bins,
        linewidth=2,
        histtype='step',
        orientation='vertical'
    )

    axs[2].text(
        x=0.75,
        y=0.95,
        s=f"n_bins = {n_bins}",
        transform=axs[2].transAxes
    )

    # axs[2].set_ylim([0, 1])

    axs[2].set_xlabel("counts", fontsize=label_fs)
    axs[2].set_ylabel("tail / total (a.u.)", fontsize=label_fs)

    fig.set_facecolor('white')

nbins_box = widgets.IntText(value=100)


def df_to_psd(
    df: pd.DataFrame, 
    peak_offset, 
    tail_onset,
    amplitudes
):
    output = df.apply(lambda x: get_bases(x, peak_idx[x.name][0], peak_offset, tail_onset))
    left_bases, right_bases = output.iloc[0,:], output.iloc[1,:]
    return generate_psd(df, left_bases, right_bases, amplitudes)
    

    

In [ ]:
def timed_plot_psd(
    df: pd.DataFrame, 
    peak_offset, 
    tail_onset,
    amplitudes,
    nbins
):
    start_time = time.perf_counter()
    plot_psd(
        df_to_psd(df, peak_offset, tail_onset, df.max()),
        nbins
    )
    
    print(
        f"Action completed in [\x1b[1;32m{(time.perf_counter() - start_time)*1000:.2f} ms\x1b[0m]."
    )

interact(
    lambda peak_offset, tail_onset, nbins: timed_plot_psd(
        df, 
        peak_offset, 
        tail_onset, 
        df.max(),
        nbins
    ),
    peak_offset = peak_offset_box,
    tail_onset = tail_onset_box,
    nbins = nbins_box
)

plt.show()